<a href="https://colab.research.google.com/github/omarelmalak/csc490-group2/blob/haris-log-det-robo-flow/Logo_Detector_combined.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLOv8 Logo Detection End-to-End Pipeline

### Step 1: Environment Setup and Library Imports

In [1]:
# Install Ultralytics library for YOLOv8 training
!pip install -q ultralytics

import os
import shutil
import glob
import yaml
import random
import xml.etree.ElementTree as ET
from collections import defaultdict
from pathlib import Path
from ultralytics import YOLO

print("Environment ready and libraries imported!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 5.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Environment ready and libraries imported!


### Step 2: Mount Google Drive
Run this cell to link your Google Drive so the script can access your zipped dataset.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Step 3: Extract Logo Dataset
Extracts your zipped dataset directly from Google Drive into the local session storage.

In [3]:
# Unzip the logo dataset from Google Drive into /content/logos
!unzip -q "/content/drive/MyDrive/csc490/LogoDet-3K.zip" -d "/content/logos"

# Inspect the extracted directory structure to verify it matches what we need
print("Extracted contents of /content/logos:")
for root, dirs, files in os.walk("/content/logos"):
    level = root.replace("/content/logos", '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    for f in files[:3]:  # Print up to 3 files as preview
        print(f"{subindent}{f}")
    if len(files) > 3:
        print(f"{subindent}... and {len(files) - 3} more files")

Streaming output truncated to the last 5000 lines.
                ... and 147 more files
            thomapyrin/
                182.xml
                72.jpg
                197.xml
                ... and 395 more files
            Apiretal/
                26.xml
                41.xml
                49.xml
                ... and 87 more files
            ringtons/
                17.xml
                23.xml
                18.jpg
                ... and 23 more files
            atamel/
                26.xml
                103.jpg
                94.jpg
                ... and 269 more files
            yuyue-1/
                72.jpg
                26.xml
                94.jpg
                ... and 181 more files
        Sports/
            neil pryde-2/
                72.jpg
                26.xml
                94.jpg
                ... and 229 more files
            spalding/
                26.xml
                94.jpg
                49.xml
                ...

### Step 3b: Download the Roboflow Sponsor Detector Dataset
Downloads `syed-fathaullah/sponsor-detector` v2 in YOLOv8 format into `/content/sponsor_rf`.

You need a free Roboflow API key (Roboflow > Settings > API Keys). Add it in Colab under the key icon (Secrets) as `ROBOFLOW_API_KEY` and enable notebook access.


In [4]:
!pip install -q roboflow

from roboflow import Roboflow
from google.colab import userdata

RF_DIR = "/content/sponsor_rf"

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
rf_project = rf.workspace("syed-fathaullah").project("sponsor-detector")
rf_dataset = rf_project.version(2).download("yolov8", location=RF_DIR, overwrite=True)

# Show the class list exactly as Roboflow defines it
with open(os.path.join(RF_DIR, "data.yaml")) as f:
    rf_yaml = yaml.safe_load(f)
rf_names = rf_yaml["names"]
if isinstance(rf_names, dict):
    rf_names = [rf_names[k] for k in sorted(rf_names, key=int)]
print(f"Roboflow classes ({len(rf_names)}): {rf_names}")

for split in ["train", "valid", "test"]:
    img_dir = os.path.join(RF_DIR, split, "images")
    if os.path.isdir(img_dir):
        print(f"{split}: {len(os.listdir(img_dir))} images")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 5.8 MB/s eta 0:00:00
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/sponsor_rf in yolov8:: 100%|██████████| 5350/5350 [00:01<00:00, 5254.80it/s]


Roboflow classes (4): ['Adidas', 'Emirates', 'Nike', 'Spotify']
train: 2488 images
valid: 125 images
test: 56 images


### Step 4: Convert Pascal XML Annotations to YOLO TXT Format
Parses the XML annotations in the "Other" folder and converts them to standard normalized YOLO `.txt` coordinate files next to each image.

In [5]:
# --- CONFIGURATION ---
SOURCE_XML_DIR = "/content/logos/LogoDet-3K/Others"

# 1. First, discover all folders (classes) in Others to build a robust, deterministic class list
# This ensures the IDs never drift between XML parsing and YAML generation.
raw_classes = sorted([d for d in os.listdir(SOURCE_XML_DIR) if os.path.isdir(os.path.join(SOURCE_XML_DIR, d))])

class_to_id = {name: i for i, name in enumerate(raw_classes)}
print(f"Deterministic mapping created for {len(raw_classes)} directories.")

def convert_coords(size, box):
    dw = 1.0 / size[0] if size[0] > 0 else 1.0
    dh = 1.0 / size[1] if size[1] > 0 else 1.0
    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0
    w = box[1] - box[0]
    h = box[3] - box[2]
    return (x * dw, y * dh, w * dw, h * dh)

# Convert each XML to TXT inside its respective directory
xml_files = glob.glob(os.path.join(SOURCE_XML_DIR, "**/*.xml"), recursive=True)
converted_count = 0
for xml_file in xml_files:
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        size = root.find('size')
        w = int(size.find('width').text)
        h = int(size.find('height').text)
        if w == 0 or h == 0:
            continue

        txt_filename = os.path.splitext(os.path.basename(xml_file))[0] + ".txt"
        txt_output_path = os.path.join(os.path.dirname(xml_file), txt_filename)

        with open(txt_output_path, 'w') as out_file:
            for obj in root.iter('object'):
                cls_name = obj.find('name').text
                # Use the immediate parent folder name as the fallback ground truth class
                parent_folder = os.path.basename(os.path.dirname(xml_file))

                # Resolve class ID using the directory name to ensure alignment
                cls_id = class_to_id.get(parent_folder, class_to_id.get(cls_name, 0))

                xmlbox = obj.find('bndbox')
                b = (
                    float(xmlbox.find('xmin').text),
                    float(xmlbox.find('xmax').text),
                    float(xmlbox.find('ymin').text),
                    float(xmlbox.find('ymax').text)
                )
                bb = convert_coords((w, h), b)
                out_file.write(f"{cls_id} {' '.join([f'{coord:.6f}' for coord in bb])}\n")
        converted_count += 1
    except Exception as e:
        pass

print(f"Successfully converted {converted_count} XML files to aligned YOLO TXT format!")

Deterministic mapping created for 371 directories.
Successfully converted 15513 XML files to aligned YOLO TXT format!


### Step 5: Define Paths & Split Dataset (80% Train, 10% Val, 10% Test)
This section processes only the "Other" folder, matches the images and converted `.txt` label files, and distributes them proportionally.

In [6]:
import os
import glob
import random
import shutil
from collections import defaultdict

# Configuration
SOURCE_IMAGES_DIR = "/content/logos/LogoDet-3K/Others"
DATASET_ROOT = "/content/logo_yolo_dataset"

# 1. Clean and recreate directory structures completely
if os.path.exists(DATASET_ROOT):
    shutil.rmtree(DATASET_ROOT)

for split_type in ['train', 'val', 'test']:
    os.makedirs(os.path.join(DATASET_ROOT, 'images', split_type), exist_ok=True)
    os.makedirs(os.path.join(DATASET_ROOT, 'labels', split_type), exist_ok=True)

# 2. Gather image files recursively
all_images = []
valid_exts = ('.png', '.jpg', '.jpeg')
for root, dirs, files in os.walk(SOURCE_IMAGES_DIR):
    for f in files:
        if f.lower().endswith(valid_exts):
            all_images.append(os.path.join(root, f))

# 3. Match image-label pairs grouped by brand folder
class_to_pairs = defaultdict(list)
for img_path in all_images:
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    img_dir = os.path.dirname(img_path)
    label_path = os.path.join(img_dir, f"{base_name}.txt")
    class_name = os.path.basename(img_dir)
    if os.path.exists(label_path):
        class_to_pairs[class_name].append((img_path, label_path))

# 4. Perform robust split and copy with class-prefixed unique filenames
segmented_count = 0
total_copied = 0

for class_name, pairs in class_to_pairs.items():
    random.seed(42)
    random.shuffle(pairs)

    n_samples = len(pairs)
    if n_samples == 0:
        continue

    val_idx = int(n_samples * 0.8)
    test_idx = int(n_samples * 0.9)

    train_pairs = pairs[:val_idx]
    val_pairs = pairs[val_idx:test_idx]
    test_pairs = pairs[test_idx:]

    if len(val_pairs) == 0 and n_samples > 1:
        val_pairs = [pairs[-1]]
        train_pairs = pairs[:-1]

    def copy_files_unique(pairs_list, split_name):
        global total_copied
        for img, lbl in pairs_list:
            img_ext = os.path.splitext(img)[1]
            base = os.path.splitext(os.path.basename(img))[0]

            # We prefix the filename with the class name to make it totally unique
            unique_name = f"{class_name}_{base}"

            dest_img = os.path.join(DATASET_ROOT, 'images', split_name, f"{unique_name}{img_ext}")
            dest_lbl = os.path.join(DATASET_ROOT, 'labels', split_name, f"{unique_name}.txt")

            shutil.copy2(img, dest_img)
            shutil.copy2(lbl, dest_lbl)
            total_copied += 1

    copy_files_unique(train_pairs, 'train')
    copy_files_unique(val_pairs, 'val')
    copy_files_unique(test_pairs, 'test')

    segmented_count += 1

print(f"=== Unique-Filename Dataset Split Completed ===")
print(f"Successfully split {segmented_count} logo classes.")
print(f"Total unique paired files copied to disk: {total_copied}")

=== Unique-Filename Dataset Split Completed ===
Successfully split 371 logo classes.
Total unique paired files copied to disk: 15533


In [7]:
import os
import glob

# Check the physical count on disk now
print("=== NEW DISK STATS ===")
for split in ['train', 'val', 'test']:
    imgs = os.listdir(f"/content/logo_yolo_dataset/images/{split}")
    lbls = os.listdir(f"/content/logo_yolo_dataset/labels/{split}")
    print(f"{split.capitalize()} Images: {len(imgs)} | Labels: {len(lbls)}")

print("\n=== Checking TeamViewer labels in train split ===")
tv_train_files = glob.glob("/content/logo_yolo_dataset/labels/train/teamviewer_*.txt")
print(f"Found {len(tv_train_files)} unique TeamViewer training files physically written to disk!")
if tv_train_files:
    with open(tv_train_files[0], 'r') as f:
        print(f"Sample label content from '{os.path.basename(tv_train_files[0])}':\n{f.read().strip()}")

=== NEW DISK STATS ===
Train Images: 12257 | Labels: 12257
Val Images: 1558 | Labels: 1558
Test Images: 1718 | Labels: 1718

=== Checking TeamViewer labels in train split ===
Found 71 unique TeamViewer training files physically written to disk!
Sample label content from 'teamviewer_81.txt':
306 0.174213 0.488889 0.214567 0.616667
306 0.646654 0.486111 0.671260 0.311111


### Step 5b: Merge the Roboflow Dataset into the YOLO Dataset
Remaps Roboflow class IDs onto the LogoDet class list and copies the files into `DATASET_ROOT`.

* Roboflow's own train/valid/test split is kept as is. Its train set already contains augmented copies (2 per source image), so re-splitting would leak near-duplicates into val/test.
* Class names are matched case-insensitively against the LogoDet folder names. Brands that don't exist in LogoDet are appended as new classes after the existing IDs, so the existing LogoDet IDs don't change.
* Use `CLASS_ALIASES` to force a Roboflow name onto a specific LogoDet folder name (check the printed mapping before training).
* Use `DROP_CLASSES` for anything you don't want, such as a generic `sponsor` class if one exists. Images containing a dropped class are skipped entirely so the dropped logo isn't learned as background.
* Run this after Step 5 (which wipes `DATASET_ROOT`) and before Step 6 (which writes the YAML from `class_to_id`).


In [8]:
RF_DIR = "/content/sponsor_rf"
DATASET_ROOT = "/content/logo_yolo_dataset"

# Optional manual fixes: {"Roboflow name": "LogoDet folder name"}
CLASS_ALIASES = {
    # "Emirates": "emirates_airline",
}
# Roboflow classes to exclude (case-insensitive)
DROP_CLASSES = {"sponsor"}

def norm(name):
    return str(name).strip().lower().replace(" ", "_").replace("-", "_")

# Reload Roboflow names in case Step 3b ran in an earlier session
with open(os.path.join(RF_DIR, "data.yaml")) as f:
    rf_names = yaml.safe_load(f)["names"]
if isinstance(rf_names, dict):
    rf_names = [rf_names[k] for k in sorted(rf_names, key=int)]

# 1. Build Roboflow ID -> merged ID mapping
existing_by_norm = {norm(n): n for n in class_to_id}
drop_norm = {norm(n) for n in DROP_CLASSES}
alias_norm = {norm(k): v for k, v in CLASS_ALIASES.items()}

rf_to_merged = {}
dropped_rf_ids = set()
print("=== Roboflow -> merged class mapping ===")
for rf_id, rf_name in enumerate(rf_names):
    key = norm(rf_name)
    if key in drop_norm:
        dropped_rf_ids.add(rf_id)
        print(f"  [{rf_id}] {rf_name!r:<25} -> DROPPED")
        continue
    if key in alias_norm:
        target = alias_norm[key]
        if target not in class_to_id:
            raise ValueError(f"Alias target {target!r} is not a LogoDet class")
        status = "alias"
    elif key in existing_by_norm:
        target = existing_by_norm[key]
        status = "matched LogoDet"
    else:
        target = key
        if target not in class_to_id:
            class_to_id[target] = len(class_to_id)
        status = "NEW class"
    rf_to_merged[rf_id] = class_to_id[target]
    print(f"  [{rf_id}] {rf_name!r:<25} -> [{class_to_id[target]}] {target} ({status})")

print(f"\nTotal classes after merge: {len(class_to_id)}")

# 2. Remap labels and copy files (Roboflow 'valid' -> our 'val')
def to_box(parts):
    """Return (cx, cy, w, h). Converts polygon rows to their bounding box if present."""
    vals = list(map(float, parts[1:]))
    if len(vals) == 4:
        return vals
    xs, ys = vals[0::2], vals[1::2]
    x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
    return [(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1]

split_map = {"train": "train", "valid": "val", "test": "test"}
stats = defaultdict(lambda: {"copied": 0, "skipped_dropped": 0, "background": 0, "boxes": 0})

for rf_split, our_split in split_map.items():
    img_dir = os.path.join(RF_DIR, rf_split, "images")
    lbl_dir = os.path.join(RF_DIR, rf_split, "labels")
    if not os.path.isdir(img_dir):
        continue
    for img_name in os.listdir(img_dir):
        if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        base, ext = os.path.splitext(img_name)
        src_lbl = os.path.join(lbl_dir, base + ".txt")

        lines = []
        if os.path.exists(src_lbl):
            with open(src_lbl) as f:
                lines = [l.split() for l in f.read().strip().splitlines() if l.strip()]

        if any(int(p[0]) in dropped_rf_ids for p in lines):
            stats[our_split]["skipped_dropped"] += 1
            continue

        out_lines = []
        for p in lines:
            cx, cy, w, h = to_box(p)
            out_lines.append(f"{rf_to_merged[int(p[0])]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

        unique_name = f"rf_{base}"
        shutil.copy2(os.path.join(img_dir, img_name),
                     os.path.join(DATASET_ROOT, "images", our_split, unique_name + ext))
        with open(os.path.join(DATASET_ROOT, "labels", our_split, unique_name + ".txt"), "w") as f:
            f.write("\n".join(out_lines) + ("\n" if out_lines else ""))

        stats[our_split]["copied"] += 1
        stats[our_split]["boxes"] += len(out_lines)
        if not out_lines:
            stats[our_split]["background"] += 1

print("\n=== Roboflow merge stats ===")
for split, s in stats.items():
    print(f"{split}: copied {s['copied']} images ({s['boxes']} boxes, {s['background']} background-only), "
          f"skipped {s['skipped_dropped']} with dropped classes")

print("\n=== Combined dataset on disk ===")
for split in ['train', 'val', 'test']:
    n_img = len(os.listdir(os.path.join(DATASET_ROOT, 'images', split)))
    n_rf = len(glob.glob(os.path.join(DATASET_ROOT, 'images', split, 'rf_*')))
    print(f"{split}: {n_img} images ({n_rf} from Roboflow)")


=== Roboflow -> merged class mapping ===
  [0] 'Adidas'                  -> [371] adidas (NEW class)
  [1] 'Emirates'                -> [372] emirates (NEW class)
  [2] 'Nike'                    -> [373] nike (NEW class)
  [3] 'Spotify'                 -> [374] spotify (NEW class)

Total classes after merge: 375

=== Roboflow merge stats ===
train: copied 2488 images (7367 boxes, 0 background-only), skipped 0 with dropped classes
val: copied 125 images (390 boxes, 0 background-only), skipped 0 with dropped classes
test: copied 56 images (158 boxes, 0 background-only), skipped 0 with dropped classes

=== Combined dataset on disk ===
train: 14745 images (2488 from Roboflow)
val: 1683 images (125 from Roboflow)
test: 1774 images (56 from Roboflow)


### Step 6: Generate Dataset Configuration YAML

In [9]:
# Generate dataset configuration YAML using the EXACT same deterministic mapping from Step 4
class_dict = {i: name for name, i in class_to_id.items()}

yaml_data = {
    'path': DATASET_ROOT,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': class_dict
}

yaml_path = os.path.join(DATASET_ROOT, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

print(f"YOLO dataset configuration file saved to: {yaml_path}")

YOLO dataset configuration file saved to: /content/logo_yolo_dataset/dataset.yaml


### Step 7: Train the YOLO Detector

Optionally clear the existing model weights:

In [10]:
import shutil
import os

target_dir = '/content/runs/detect/logo_detection/yolov8_logos'

if os.path.exists(target_dir):
    # Forcefully delete the directory and all files inside
    shutil.rmtree(target_dir)
    print(f"Successfully force-deleted: {target_dir}")
else:
    print(f"Directory does not exist: {target_dir}")

Directory does not exist: /content/runs/detect/logo_detection/yolov8_logos


In [12]:
print("Initializing pre-trained YOLOv8 nano model...")
model = YOLO('yolov8n.pt')

# Train the model with checkpoint saving enabled
# By default, YOLOv8 saves checkpoints in {project}/{name}/weights/last.pt and best.pt
results = model.train(
    data=yaml_path,
    epochs=25,
    imgsz=640,
    project='/content/drive/MyDrive/csc490/logo_detection',   # was 'logo_detection'
    batch=16,
    device=0,          # Set to 0 if utilizing a GPU runtime
    save=True,         # This guarantees that checkpoint weights (.pt files) are saved
    name='yolov8_logos'
)

print("Training process finished!")

# --- HOW TO RESUME FROM A CHECKPOINT ---
# If your training is interrupted, you can resume it by uncommenting and running the code below:
# model = YOLO('/content/logo_detection/yolov8_logos/weights/last.pt')
# model.train(resume=True)

Initializing pre-trained YOLOv8 nano model...
Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/logo_yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yol

### Step 8: Inference on Custom Test Images
This section sets up a local test directory where you can upload your own custom images. It runs the trained YOLO model on those images, uses OpenCV (`cv2`) to draw bounding boxes and class labels directly on the images, and outputs both the visual image results and the corresponding bounding box coordinate `.txt` files.

In [13]:
import cv2
from google.colab import files

# --- CONFIGURATION ---
TEST_UPLOAD_DIR = "/content/custom_test_images"
INFERENCE_OUTPUT_DIR = "/content/custom_test_results"

# Prepare folders
os.makedirs(TEST_UPLOAD_DIR, exist_ok=True)
os.makedirs(INFERENCE_OUTPUT_DIR, exist_ok=True)

print(f"Test upload directory is ready at: {TEST_UPLOAD_DIR}")
print(f"Inference results will be saved to: {INFERENCE_OUTPUT_DIR}")

Test upload directory is ready at: /content/custom_test_images
Inference results will be saved to: /content/custom_test_results


#### Run this cell to upload your custom logo images to the test folder:

In [21]:
uploaded = files.upload()
for filename in uploaded.keys():
    shutil.move(filename, os.path.join(TEST_UPLOAD_DIR, filename))
print(f"Uploaded {len(uploaded)} images to {TEST_UPLOAD_DIR}")

Saving GFAFU7W4ERDMRD35BVMVZ7QO6M.avif to GFAFU7W4ERDMRD35BVMVZ7QO6M.avif
Uploaded 1 images to /content/custom_test_images


#### Run the detector, draw bounding boxes, and save outputs:

In [24]:
# 1. Load the best trained model (using trained weights if available, fallback to basic nano)
model_path = '/content/logo_detection/yolov8_logos/weights/best.pt'
if os.path.exists(model_path):
    test_model = YOLO(model_path)
    print(f"Loaded custom trained model weights from: {model_path}")
else:
    test_model = model
    print("Using the currently active trained model instance.")

# 2. Get all images in test folder
test_images = []
for ext in ('*.png', '*.jpg', '*.jpeg', '*.JPG', '*.JPEG', '*.PNG'):
    test_images.extend(glob.glob(os.path.join(TEST_UPLOAD_DIR, ext)))

if len(test_images) == 0:
    print(f"No images found in {TEST_UPLOAD_DIR}. Please upload some files in the step above.")
else:
    # 3. Perform prediction and annotate
    for img_path in test_images:
        base_name = os.path.splitext(os.path.basename(img_path))[0]

        # Load image with OpenCV
        img = cv2.imread(img_path)
        h, w, _ = img.shape

        # Run YOLO inference with a lowered confidence threshold of 0.10 to help catch detections
        results = test_model.predict(source=img_path, conf=0.10, verbose=False)

        txt_output_path = os.path.join(INFERENCE_OUTPUT_DIR, f"{base_name}_predictions.txt")
        detections_found = []

        with open(txt_output_path, 'w') as txt_file:
            for r in results:
                boxes = r.boxes
                for box in boxes:
                    # Get coordinate values
                    coords = box.xyxy[0].tolist() # Left, Top, Right, Bottom
                    cls_id = int(box.cls[0].item())
                    conf = float(box.conf[0].item())
                    cls_name = test_model.names.get(cls_id, f"Class_{cls_id}")

                    detections_found.append((cls_name, conf))
                    x1, y1, x2, y2 = map(int, coords)

                    # Save coordinates in normalized format to the .txt file: [class_id center_x center_y width height]
                    dw = 1.0 / w
                    dh = 1.0 / h
                    cx = (x1 + x2) / 2.0 * dw
                    cy = (y1 + y2) / 2.0 * dh
                    bw = (x2 - x1) * dw
                    bh = (y2 - y1) * dh
                    txt_file.write(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")

                    # Draw bounding box on the image using OpenCV
                    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)

                    # Add class and confidence label
                    label_str = f"{cls_name}: {conf:.2f}"
                    cv2.putText(img, label_str, (x1, max(y1 - 10, 15)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        # Save the annotated image
        annotated_img_path = os.path.join(INFERENCE_OUTPUT_DIR, f"{base_name}_result.jpg")
        cv2.imwrite(annotated_img_path, img)

        if len(detections_found) > 0:
            print(f"\nProcessed {os.path.basename(img_path)} -> Detections saved to {INFERENCE_OUTPUT_DIR}")
            print("Detections:")
            for cls_name, conf in detections_found:
                print(f" - {cls_name}: {conf:.4f} confidence")
        else:
            print(f"Processed {os.path.basename(img_path)} -> No logos detected above confidence threshold 0.10. {txt_output_path} is empty.")

Using the currently active trained model instance.

Processed Ronaldo_vs._FC_Schalke_04_(16854146922).jpg -> Detections saved to /content/custom_test_results
Detections:
 - emirates: 0.6647 confidence
 - adidas: 0.6382 confidence
 - nike: 0.2779 confidence

Processed images (2).jpeg -> Detections saved to /content/custom_test_results
Detections:
 - nike: 0.4207 confidence
 - spotify: 0.4130 confidence
 - emirates: 0.2996 confidence
 - spotify: 0.2923 confidence
 - spotify: 0.2188 confidence

Processed images (1).jpeg -> Detections saved to /content/custom_test_results
Detections:
 - nike: 0.5358 confidence
 - nike: 0.1933 confidence

Processed images.jpeg -> Detections saved to /content/custom_test_results
Detections:
 - spotify: 0.7757 confidence
 - spotify: 0.5482 confidence
 - nike: 0.5159 confidence
 - nike: 0.2702 confidence
 - spotify: 0.1570 confidence
 - spotify: 0.1489 confidence
 - spotify: 0.1232 confidence


### Step 9: Diagnostic Check - Inspect 'teamviewer' Dataset Integrity
Run this cell to count how many images and label files actually exist for `teamviewer` in your source directory, and see what the bounding box files contain.

In [ ]:
import os
import glob

# --- Unified Single Dataset Validation Cell (Diagnostic Mode) ---
TARGET_BRAND = "teamviewer"
print(f"=== VALIDATING DATASET INTEGRITY FOR '{TARGET_BRAND}' ===\n")

# 1. Resolve Expected Class ID from deterministic map
expected_id = class_to_id.get(TARGET_BRAND, 306) if 'class_to_id' in globals() else 306
print(f"-> Expected Class ID in dataset.yaml: {expected_id}\n")

# 2. Let's inspect the files on disk vs the labels inside them
DATASET_ROOT = "/content/logo_yolo_dataset"

for split in ['train', 'val', 'test']:
    img_dir = os.path.join(DATASET_ROOT, 'images', split)
    lbl_dir = os.path.join(DATASET_ROOT, 'labels', split)

    # Count physical files copied by Step 5
    physical_label_files = []
    if os.path.exists(lbl_dir):
        # Filter files that belong to teamviewer (since we know teamviewer file names from original folder)
        # Let's inspect the actual lines written inside whatever label files we have in this split
        physical_label_files = glob.glob(os.path.join(lbl_dir, "*.txt"))

    print(f"--- Split '{split}': ---")
    print(f"  * Total physical label files (.txt) in this split folder: {len(physical_label_files)}")

    # Read what class IDs are actually written inside these labels
    found_ids_in_split = set()
    matching_expected_id_count = 0
    sample_lines = []

    for txt_file in physical_label_files:
        with open(txt_file, 'r') as f:
            content = f.read().strip()
            if content:
                for line in content.split('\n'):
                    parts = line.split()
                    if parts:
                        cls_id = parts[0]
                        found_ids_in_split.add(cls_id)
                        if cls_id == str(expected_id):
                            matching_expected_id_count += 1
                            if len(sample_lines) < 3:
                                sample_lines.append((os.path.basename(txt_file), line))

    print(f"  * Distinct Class IDs actually written inside these text files: {sorted(list(found_ids_in_split))[:15]}")
    print(f"  * Files containing the target class ID '{expected_id}': {matching_expected_id_count}")
    if sample_lines:
         print("  * Sample matching lines:")
         for fname, s_line in sample_lines:
              print(f"     - {fname}: {s_line}")
    print("")

=== VALIDATING DATASET INTEGRITY FOR 'teamviewer' ===

-> Expected Class ID in dataset.yaml: 306

--- Split 'train': ---
  * Total physical label files (.txt) in this split folder: 12257
  * Distinct Class IDs actually written inside these text files: ['0', '1', '10', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '11', '110']
  * Files containing the target class ID '306': 108
  * Sample matching lines:
     - teamviewer_6.txt: 306 0.585239 0.478643 0.754678 0.414573
     - teamviewer_6.txt: 306 0.113306 0.581658 0.189189 0.670854
     - teamviewer_72.txt: 306 0.446629 0.736626 0.449438 0.078189

--- Split 'val': ---
  * Total physical label files (.txt) in this split folder: 1558
  * Distinct Class IDs actually written inside these text files: ['0', '1', '10', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '11', '110']
  * Files containing the target class ID '306': 13
  * Sample matching lines:
     - teamviewer_22.txt: 306 0.635189 0.47

### Step 9: Diagnostic Check - Inspect Training Results & Validate Model
Run this cell to find the exact path of your newly trained weights and run a standard validation batch. This verifies if the model actually learned anything during those 5 epochs.

In [16]:
import glob
import os
from ultralytics import YOLO

# 1. Automatically find the latest trained weights under logo_detection directory
weights_paths = glob.glob("/content/runs/detect/logo_detection/*/weights/best.pt") + glob.glob("/content/logo_detection/*/weights/best.pt")

if weights_paths:
    # Sort by modification time to get the newest run's weights
    latest_weights = max(weights_paths, key=os.path.getmtime)
    print(f"[SUCCESS] Found trained weights at: {latest_weights}")

    # Load the trained model
    diagnostic_model = YOLO(latest_weights)

    # 2. Run standard validation to see overall model performance metrics (mAP50, Precision, Recall)
    print("\n--- Running standard validation on validation dataset split ---")
    val_results = diagnostic_model.val()
else:
    print("[WARNING] No trained 'best.pt' weights found anywhere! Your model has not saved training weights successfully or the path is wrong.")

[WARNING] No trained 'best.pt' weights found anywhere! Your model has not saved training weights successfully or the path is wrong.
